# Module 8 · Function calling and tool integration

The model never runs your code. It returns a **function call** (name + arguments); *you* run it and send the result back.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |
| `OPENWEATHER_API_KEY` | OpenWeatherMap key (free plan) | https://home.openweathermap.org/api_keys |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai requests langchain-core langchain-google-genai

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) "))
# Change to "gemini-2.5-flash" if your key does not have access to the newer model.
MODEL = "gemini-3.5-flash-lite"
print('Ready. Model =', MODEL)

## 1 · Single tool: live weather
**New OpenWeatherMap key?** It can take 10–120 minutes to activate; until then you'll see a 401 error.

In [ ]:
import requests
OWM_KEY = get_secret("OPENWEATHER_API_KEY", hint="(from home.openweathermap.org/api_keys) ")

def get_current_temperature(location: str) -> float:
    """Return the current temperature in Celsius for a city, e.g. 'Gurugram,IN'."""
    r = requests.get("https://api.openweathermap.org/data/2.5/weather",
                     params={"q": location, "appid": OWM_KEY, "units": "metric"}, timeout=15)
    data = r.json()
    if r.status_code != 200:
        raise RuntimeError(f"OpenWeather error {r.status_code}: {data.get('message')}")
    return data["main"]["temp"]

weather_decl = {
    "name": "get_current_temperature",
    "description": "Gets the current temperature for a given location.",
    "parameters": {"type": "object",
                   "properties": {"location": {"type": "string", "description": "City name, e.g. San Francisco"}},
                   "required": ["location"]},
}
config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[weather_decl])])

resp = client.models.generate_content(model=MODEL, contents="What's the temperature in Gurugram, Haryana?", config=config)
part = resp.candidates[0].content.parts[0]
print("Model asked for:", part.function_call)
if part.function_call:
    args = dict(part.function_call.args)
    print("Result:", get_current_temperature(**args), "°C")

## 2 · The full loop: send the result back so the model can answer in words

In [ ]:
contents = [types.Content(role="user", parts=[types.Part(text="Is it warm in Delhi right now? Give the temperature.")])]
resp = client.models.generate_content(model=MODEL, contents=contents, config=config)
call = resp.candidates[0].content.parts[0].function_call
if call:
    contents.append(resp.candidates[0].content)                       # the model's tool request
    try:
        result = {"result": get_current_temperature(**dict(call.args))}
    except Exception as e:
        result = {"error": str(e)}                                    # tell the model the tool failed
    contents.append(types.Content(role="user", parts=[types.Part.from_function_response(name=call.name, response=result)]))
    final = client.models.generate_content(model=MODEL, contents=contents, config=config)
    print(final.text)
else:
    print(resp.text)

## 3 · Multiple tools, one question (manual loop)

In [ ]:
def add(a: int, b: int) -> int:
    print(f"  executing add({a}, {b})"); return a + b
def multiply(a: int, b: int) -> int:
    print(f"  executing multiply({a}, {b})"); return a * b
available = {"add": add, "multiply": multiply}

def decl(name, desc):
    return types.FunctionDeclaration(name=name, description=desc, parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={"a": types.Schema(type=types.Type.INTEGER, description="First integer"),
                    "b": types.Schema(type=types.Type.INTEGER, description="Second integer")},
        required=["a", "b"]))

cfg = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[decl("add", "Adds two integers."), decl("multiply", "Multiplies two integers.")])],
                                  automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
msgs = [types.Content(role="user", parts=[types.Part(text="What is 3 * 12? Also, what is 11 + 49?")])]
r1 = client.models.generate_content(model=MODEL, contents=msgs, config=cfg)
msgs.append(r1.candidates[0].content)
outs = []
for p in r1.candidates[0].content.parts:
    if p.function_call:
        fn = available[p.function_call.name]
        outs.append(types.Part.from_function_response(name=p.function_call.name, response={"result": fn(**dict(p.function_call.args))}))
msgs.append(types.Content(role="user", parts=outs))
print(client.models.generate_content(model=MODEL, contents=msgs, config=cfg).text)

## 4 · Automatic function calling
Pass plain Python functions (with type hints and a docstring) and the SDK runs the whole loop for you.

In [ ]:
r = client.models.generate_content(model=MODEL, contents="What is 7 times 6, plus 100?",
                                   config=types.GenerateContentConfig(tools=[add, multiply]))
print(r.text)
print("calls made:", [h.parts[0].function_call.name for h in (r.automatic_function_calling_history or []) if h.parts and h.parts[0].function_call])

## 5 · Tool modes: AUTO · ANY · NONE
`AUTO` lets the model decide, `ANY` forces a tool call, `NONE` forbids it.

In [ ]:
for mode in ["AUTO", "ANY", "NONE"]:
    cfg = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[decl("add", "Adds two integers.")])],
        tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode=mode)),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    r = client.models.generate_content(model=MODEL, contents="Say hello to me.", config=cfg)
    p = r.candidates[0].content.parts[0]
    print(mode.ljust(5), "->", "FUNCTION CALL " + p.function_call.name if p.function_call else "text: " + (p.text or "")[:60])

## 6 · The same tools through LangChain

In [ ]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

@tool
def lc_add(a: int, b: int) -> int:
    """Add two integers"""
    return a + b

@tool
def lc_multiply(a: int, b: int) -> int:
    """Multiply two integers"""
    return a * b

llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=os.environ["GEMINI_API_KEY"]).bind_tools([lc_add, lc_multiply])
reg = {"lc_add": lc_add, "lc_multiply": lc_multiply}
ai = llm.invoke("What is 12 plus 8, and what is 7 times 6?")
for tc in ai.tool_calls:
    print(tc["name"], tc["args"], "->", reg[tc["name"]].invoke(tc["args"]))